# Mining the Sundown County Data for Local Lynching Reports

The following notebook covers how I mined my sundown county data for potential references to local lynchings. These steps utilize the data compiled in 01_compile_sundown_county_data.ipynb. It reviews this data more closely, seeking out references to local lynchings by searching for instances of the known victims' names.

The primary goal of this step: to build a list of potential references to local lynchings that I could review by hand and use as training data for a language model (to be used to identify similar newspaper text elsewhere). As you'll see, this goal was achieved. Hooray!

Also, full disclosure: this code was partially/collaboratively written with Codex and Claude Code. My process is beginning to look more and more like this: I generate pseudocode then prompt LLMs to generate Python. Then I review, edit, refactor, and implement if it looks logical and effective.

I begin, as always, by loading the necessary libraries and dataframes. I'm using a portable SSD where my data is stored (the base path below). I'm also using two .csv files, loaded as lynch_records and chron_am. I've previously described these data in 01_compile_sundown_county_data.ipynb, but in a nutshell: lynch_records is two combined inventories of lynchings that occurred in the 19th and 20th centuries, and chron_am contains information about the newspapers in Chronicling America.

In [ ]:
from pathlib import Path
from collections import defaultdict
from bisect import bisect_left, bisect_right
import pandas as pd
import re
import os
import bisect
import multiprocessing as mp
from functools import partial
from concurrent.futures import ProcessPoolExecutor

base = Path('/Volumes/t5_evo_8tb/ChronAm/sundown_town_data')
lynch_records = pd.read_csv('combined_inventories.csv')
chron_am = pd.read_csv('chron_am_newspapers.csv')

In 01_compile_sundown_county_data.ipynb, I did a lot of normalizing and merging of dataframes in order to find the subset of Chronicling America that represented newspapers from counties where lynchings occurred (sundown counties). In this step, however, I quickly realized I had to redo and expand on those normalizing and merging steps.

I again had to remove 'county' and 'parish' strings from the County column in chron_am. I had to link newspapers to their respective counties using the LCCN codes. I had to build an index of cases (by victim), newspapers (by LCCN), and years (years of available digitized newspaper data per newspaper per victim in my sundown county subset of Chronicling America).

The result of these steps was essentially a fast lookup system, allowing me to navigate my directories more easily in later steps.

In [ ]:
def norm_text(x):
    return str(x).strip().casefold()

def clean_county(x):
    s = str(x).strip()
    s = re.sub(r'\b(county|parish)\b', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+', ' ', s).strip()
    return s.casefold()

def norm_lccn(x):
    return str(x).strip().casefold()

chron = chron_am.copy()
chron['county_key'] = chron['County'].map(clean_county)
chron['lccn_key'] = chron['LCCN'].map(norm_lccn)

county_to_lccn = chron.groupby('county_key')['lccn_key'].agg(set).to_dict()

victim_lccn_years = defaultdict(lambda: defaultdict(list))

for victim_dir in base.iterdir():
    if not victim_dir.is_dir():
        continue
    victim_key = norm_text(victim_dir.name)

    for sn_dir in victim_dir.iterdir():
        if not sn_dir.is_dir():
            continue
        lccn_key = norm_lccn(sn_dir.name)

        for year_dir in sn_dir.iterdir():
            if not year_dir.is_dir():
                continue
            try:
                year_val = int(year_dir.name)
                victim_lccn_years[victim_key][lccn_key].append(year_val)
            except ValueError:
                pass

for v in victim_lccn_years:
    for l in victim_lccn_years[v]:
        victim_lccn_years[v][l].sort()

This code evaluates each row in lynch_records by matching the victim and county in that row against the lookup structures built earlier, then checking whether any newspaper associated with both that victim and county has digitized coverage for the target year. For each record, it determines whether the exact year is covered, counts how many available newspaper years fall before and after that year, and records which LCCN(s) matched; if the year is invalid or no matching newspapers are found, it returns default “no coverage” values. The resulting metrics are then added back to lynch_records as new columns, with the year-count columns stored as nullable integers.

With this lookup system in place, I next turned to the lynch_records dataframe. I needed to take the year of lynchings and double-check whether newspapers in my data had digitized coverage for that given year. For more context: in 01_compile_sundown_county_data.ipynb, I created an eight-year window, capturing all county coverage per case within four years prior to the incident, the year of the incident, and three years afterwards, but I neglected to count which cases had coverage before, during, and after. All I had was "the window." So, I needed to figure out precisely which cases had coverage from the year of occurrence. This was important to lynching coverage because, of course, there wouldn't be coverage of these events in the years prior to their occurrence, but also, most likely, coverage would be most pronounced within the year that they occurred–that they would be breaking news, so to speak.

I therefore decided to count year coverage for each case–counting years prior to the incident, years of the incident, and years afterwards. To do this, I used the compute_row() function below. It reviews each row in lynch_records in conjunction with my lookup index of sundown county data. It then adds to lynch_records, labeling year-of coverage as either 'yes' or 'no', then counting prior_years and later_years.

In [ ]:
def compute_row(row):
    victim_key = norm_text(row['victim'])
    county_key = clean_county(row['county'])

    try:
        target_year = int(row['year'])
    except Exception:
        return pd.Series({
            'year_coverage': 'no',
            'prior_years': 0,
            'later_years': 0,
            'matched_lccn': pd.NA
        })

    county_lccns = county_to_lccn.get(county_key, set())
    victim_map = victim_lccn_years.get(victim_key, {})

    matched_lccns = sorted(county_lccns.intersection(victim_map.keys()))
    if not matched_lccns:
        return pd.Series({
            'year_coverage': 'no',
            'prior_years': 0,
            'later_years': 0,
            'matched_lccn': pd.NA
        })

    years = []
    for lccn in matched_lccns:
        years.extend(victim_map[lccn])
    years = sorted(set(years))

    left = bisect_left(years, target_year)
    right = bisect_right(years, target_year)

    return pd.Series({
        'year_coverage': 'yes' if right > left else 'no',
        'prior_years': left,
        'later_years': len(years) - right,
        'matched_lccn': ';'.join(matched_lccns)
    })

lynch_records[['year_coverage', 'prior_years', 'later_years', 'matched_lccn']] = (
    lynch_records.apply(compute_row, axis=1)
)

lynch_records['prior_years'] = lynch_records['prior_years'].astype('Int64')
lynch_records['later_years'] = lynch_records['later_years'].astype('Int64')

Quick pause at this point. I was running these processes on the entirety of cases in the combined lynching inventories–not really necessary, but it's just how my workflow started this time around. I only needed to be looking at cases with county-level newspaper coverage, so I created a quick filter–a mask on the matched_lccn column, removing any rows with empty values, therefore keeping only cases with county-level newspaper data.

In [ ]:
mask = (
    lynch_records['matched_lccn'].notna()
    & lynch_records['matched_lccn'].astype(str).str.strip().ne('')
)

local_coverage_cases = lynch_records.loc[mask].copy()
local_coverage_cases.to_csv('local_coverage_cases.csv', index=False)

I now had labeled case data, telling me if I had any county-level coverage from the year of incident, and a system of information wherein I could mine this coverage for reference to the incident. Next I needed a way to identify references to these cases. This is not something entirely new to me. In my previous work curating the Dataset of U.S. Lynching Reports (DUSLR), I used victim names as a potential indicator of reference to lynching cases. This worked quite well then, so I went with it again here.

One lesson from that previous work was that my success would increase if I accounted for possible OCR errors in victim names. There are a lot of ways to try to correct OCR, but none are straightforward. I tend to opt for a less-is-more approach, meaning, in this case, that I'm not going to try to correct OCR wholesale across my data. Instead, I built a system to simply identify potential OCR variants in victim names and mine all instances of the victim names plus the potential variants.

This system is outlined in the functions below. In a nutshell, it does the following:
- for any victim names (first, middle, or last) that are over three characters long, it builds a list of variants of each name with one character off–a wildcard character. So, for example, this method will catch 'John Smirh', Juhn Smith', and 'Jonn Snith' as potential instances of 'John Smith'. It will ignore three-letter names since they can be overcounted by three-letter words–for example, 'rod', 'mod', 'cod', 'pod', and 'sod' would all count as 'Tod', which could clutter results.
- Splits each victim name into adjacent word pairs, using the compiled regex wildcard variants as potential stand-ins or appearances of the given name. So, this method won't catch 'John changed his last name to Smith'–it will only count name instances where 'John' and 'Smith' (or their one-character-off variations) appear as the pair 'John Smith'.
- Takes these potential variations and applies them to OCR text, allowing next-level name searches to capture the potential OCR variations.
- Tokenizes the OCR text and records the name positions so I can later map the matches back to their original positions within the text.

In [ ]:
def _token_variants(token: str) -> str:
    variants = [re.escape(token)]
    if len(token) >= 3:
        for i in range(len(token)):
            variants.append(re.escape(token[:i]) + '.' + re.escape(token[i + 1:]))
    variants = list(dict.fromkeys(variants))
    return '(?:' + '|'.join(variants) + ')'

def _build_name_fix_patterns(victim_name: str):
    parts = [p for p in str(victim_name).split() if p]
    patterns = []
    for i in range(len(parts) - 1):
        first_name = parts[i]
        second_name = parts[i + 1]

        first_pattern = _token_variants(first_name)
        second_pattern = _token_variants(second_name)

        pat = re.compile(
            rf'({first_pattern})\W*({second_pattern})',
            flags=re.IGNORECASE
        )
        repl = f' {first_name} {second_name} '
        patterns.append((pat, repl))
    return patterns

def _fix_names_for_ocr(text: str, name_fix_patterns) -> str:
    for pat, repl in name_fix_patterns:
        text = pat.sub(repl, text)
    return ' '.join(text.split())

def _word_spans(text: str):
    matches = list(re.finditer(r'\S+', text))
    words = [m.group(0) for m in matches]
    spans = [m.span() for m in matches]
    starts = [s for s, _ in spans]
    return words, spans, starts

I then needed some helpers to also capture clippings surrounding the victim names. I thought these clippings would be helpful for downstream assessment and analyses. I chose to capture 100 words before and after the victim names–a rough estimate, but adjustable if need be and enough to get a sense of the context.

In [ ]:
def _char_to_word_idx(pos: int, spans, starts) -> int:
    i = bisect.bisect_right(starts, pos) - 1
    if i < 0:
        return 0
    n = len(spans)
    while i + 1 < n and spans[i][1] <= pos:
        i += 1
    return min(i, n - 1)

def _extract_clippings(text: str, victim_pattern: re.Pattern, window_words: int = 100):
    matches = list(victim_pattern.finditer(text))
    if not matches:
        return []

    words, spans, starts = _word_spans(text)
    if not words:
        return []

    out = []
    for m in matches:
        s_char, e_char = m.span()
        s_idx = _char_to_word_idx(s_char, spans, starts)
        e_idx = _char_to_word_idx(max(s_char, e_char - 1), spans, starts)

        left = max(0, s_idx - window_words)
        right = min(len(words), e_idx + 1 + window_words)

        out.append(
            {
                'matched_text': m.group(0),
                'clipping': ' '.join(words[left:right]),
            }
        )
    return out

Then I needed to build some deployable functions to run these searches on my data. As a final output, I wanted a new dataframe with one row per instance where a victim's name was used (including potential OCR variations) within the newspapers of their county in the year of their lynching.

My data directories also maintain some useful information about each page. That is, they're nested directories with subdirectories per year, month, day, edition, and page sequence per newspaper. I explained this in greater detail in 01_compile_sundown_county_data.ipynb, but suffice it to say that these nested directories also tell you which precise page of the newspaper you're mining from. To preserve this information, I factored it into the functions below, saving the nested directory info as values in respective columns.

In [ ]:
def _scan_one_target(target, base_dir: str, window_words: int):
    victim, matched_lccn, year = target
    year_dir = Path(base_dir) / victim / matched_lccn / year
    if not year_dir.is_dir():
        return []

    name_fix_patterns = _build_name_fix_patterns(victim)
    victim_pattern = re.compile(re.escape(victim), flags=re.IGNORECASE)
    rows = []

    for root, _, files in os.walk(year_dir):
        for fn in files:
            if fn.lower() != 'ocr.txt':
                continue

            ocr_path = Path(root) / fn
            try:
                raw_text = ocr_path.read_text(encoding='utf-8', errors='ignore')
            except OSError:
                continue

            text = _fix_names_for_ocr(raw_text, name_fix_patterns)
            clips = _extract_clippings(text, victim_pattern=victim_pattern, window_words=window_words)
            if not clips:
                continue

            rel = ocr_path.relative_to(year_dir).parts
            month = rel[0] if len(rel) > 0 else None
            date = rel[1] if len(rel) > 1 else None
            ed = rel[2] if len(rel) > 2 else None
            seq = rel[3] if len(rel) > 3 else None

            for c in clips:
                rows.append(
                    {
                        'victim': victim,
                        'matched_lccn': matched_lccn,
                        'year': year,
                        'month': month,
                        'date': date,
                        'ed': ed,
                        'seq': seq,
                        'ocr_path': str(ocr_path),
                        'matched_text': c['matched_text'],
                        'clipping': c['clipping'],
                    }
                )

    return rows


def search_victim_mentions_with_clippings(
    df: pd.DataFrame,
    base_dir,
    max_workers: int = None,
    chunksize: int = 20,
    window_words: int = 100,
) -> pd.DataFrame:
    base_dir = str(Path(base_dir))

    work = df.copy()
    work['year_coverage'] = work['year_coverage'].astype(str).str.strip().str.lower()
    work = work[work['year_coverage'] == 'yes'].copy()

    work['victim'] = work['victim'].astype(str).str.strip()
    work['matched_lccn'] = work['matched_lccn'].astype(str).str.strip()
    work['year'] = pd.to_numeric(work['year'], errors='coerce').astype('Int64')
    work = work.dropna(subset=['victim', 'matched_lccn', 'year']).copy()
    work['year'] = work['year'].astype(int).astype(str)

    targets = list(
        work[['victim', 'matched_lccn', 'year']]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )

    targets = [t for t in targets if (Path(base_dir) / t[0] / t[1] / t[2]).is_dir()]

    if not targets:
        return pd.DataFrame(
            columns=[
                'victim', 'matched_lccn', 'year', 'month', 'date', 'ed', 'seq',
                'ocr_path', 'matched_text', 'clipping'
            ]
        )

    worker = partial(_scan_one_target, base_dir=base_dir, window_words=window_words)
    all_rows = []

    executor_kwargs = {'max_workers': max_workers}
    if os.name != 'nt':
        executor_kwargs['mp_context'] = mp.get_context('fork')

    with ProcessPoolExecutor(**executor_kwargs) as ex:
        for batch in ex.map(worker, targets, chunksize=chunksize):
            if batch:
                all_rows.extend(batch)

    return pd.DataFrame(all_rows)

And now I was ready to run it all!

In [ ]:
results_df = search_victim_mentions_with_clippings(
     local_coverage_cases,
     base_dir=base,
     max_workers=12,
     chunksize=25,
     window_words=100
 )
results_df.to_csv('victim_mention_results.csv', index=False)

It worked! And only took about seven minutes of runtime. After a quick review of the results, however, I noticed I had made one misstep. Lots of cases were from 'Unknown' victims who were labeled 'unknown', meaning that my OCR name variation and mining process had looked for 'unknown' in the corresponding county-level data. This would be unhelpful since it's unlikely all those 'unknown' instances had anything to do with lynching cases.

I therefore had to remove all those 'Unknown' rows. It did give me something to think about, though–a potential downstream use. I could take the hand-keyed results of these local case reports and potentially use them to identify victims who were previously unknown. But that's for some later work. I will try to come back to it.

In [ ]:
results_df = results_df[results_df['victim'] != 'Unknown'].copy()
results_df.to_csv('victim_mention_results.csv', index=False)

As a quick way to review the results, I wanted to see how many cases had potential data. I did this by counting the number of unique victim name values. The result: 115 potential cases with local coverage.

In [ ]:
results_df['victim'].nunique()

I also wanted to see the number of instances victim names were potentially mentioned (i.e., the number of clippings and name instances mined through my process). The result: 1,268 potential references to local cases.

In [ ]:
len(results_df)

Last thing: I wanted an easy way to review the full pages of these cases in both OCR text and image formats. To do this, I used all the relevant data to build the Chronicling America URLs to the given pages. That way, I could review each full page by following each row's corresponding link.

In [ ]:
seq_no = results_df['seq'].astype(str).str.extract(r'(\d+)', expand=False)
victim_q = (
    results_df['victim']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '+', regex=True)
)

month = results_df['month'].astype(str).str.zfill(2)
day = results_df['date'].astype(str).str.zfill(2)

results_df['chron_am_url'] = (
    'https://www.loc.gov/resource/'
    + results_df['matched_lccn'].astype(str)
    + '/'
    + results_df['year'].astype(str)
    + '-'
    + month
    + '-'
    + day
    + '/'
    + results_df['ed'].astype(str)
    + '/?sp='
    + seq_no.fillna('')
    + '&q='
    + victim_q
    + '&st=text&r=0,0,0,0'
)

results_df.to_csv('victim_mention_results.csv', index=False)

And that was it, I now had a new .csv (victim_mention_results.csv) that contained rows per potential victim names in newspapers from their respective counties in the year of their lynchings. In the next step, I'll be reviewing this data by hand. The hope is it can yield a training dataset of local coverage to tell me more about how sundown towns and counties reported on local cases of racial violence.